# 03 - Method B: Predictive Regression with Ridge

**Inertia v2 - Factor Regimes** - Sprint 2 (V3 spec)

Predictive regression of next-month factor return on lagged factor returns, rolling factor volatilities, and **macro state variables (VIX, term spread, credit spread)**. Fit with RidgeCV (5-fold cross-validated L2 penalty), features standardized per training window. Predicted return is mapped to a favorable probability via $\sigma(\hat{r}/\hat{\sigma}_{\text{train}})$ where $\sigma$ is the logistic function.

Weight rule: **long-flat** ($w_t = 1$ if $P_t > 0.5$, else $0$).

This is variant V3 from `scripts/explore_method_b.py` (the parallel-agent winner).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.data import build_factor_panel, FF5_FACTORS
from lib.methods import fit_predict_ridge_oos
from lib.backtest import prob_to_weight, apply_weights, static_factor_returns
from lib.evaluation import perf_stats
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    yearly_xticks, legend_below, bar_value_labels,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '1990-01-01'

## 1. Run Method B on all five factors

In [2]:
panel = build_factor_panel(include_macro=True)
macro_cols = [c for c in ['vix','vix_chg','term_spread','credit_spread'] if c in panel.columns]
factor_features = ([f'lag1_{f}' for f in FF5_FACTORS] +
                   [f'vol6_{f}' for f in FF5_FACTORS] +
                   macro_cols)
print(f'Features ({len(factor_features)}): {factor_features}')

# Use 2000-02 OOS start so the macro panel (FRED post-1990) has enough lead-in
OOS_START_HERE = '2000-02-01'

probs = {}
for f in FF5_FACTORS:
    print(f'Fitting Ridge for {f}...')
    probs[f] = fit_predict_ridge_oos(panel, f, factor_features,
                                       oos_start=OOS_START_HERE, refit_months=12, min_train=120)
P = pd.DataFrame(probs)
print(f'\nOOS shape: {P.shape}, range {P.index.min().date()} -> {P.index.max().date()}')
P.describe().T[['mean','std','min','max']]


Features (14): ['lag1_MKT_RF', 'lag1_SMB', 'lag1_HML', 'lag1_RMW', 'lag1_CMA', 'vol6_MKT_RF', 'vol6_SMB', 'vol6_HML', 'vol6_RMW', 'vol6_CMA', 'vix', 'vix_chg', 'term_spread', 'credit_spread']
Fitting Ridge for MKT_RF...


Fitting Ridge for SMB...


Fitting Ridge for HML...


Fitting Ridge for RMW...


Fitting Ridge for CMA...



OOS shape: (312, 5), range 2000-02-29 -> 2026-01-31


,mean,std,min,max
MKT_RF,0.5329,0.0510,0.2444,0.7266
SMB,0.5279,0.0816,0.3517,0.9760
HML,0.5093,0.0658,0.2421,0.7181
RMW,0.5145,0.0935,0.0018,0.7291
CMA,0.5356,0.0748,0.3598,0.9053


## 2. Regime probability time series

Ridge probabilities are tighter around 0.5 because Ridge predictions are small after L2 shrinkage. The signal still varies enough to drive timing decisions.

In [3]:
fig, axes = plt.subplots(5, 1, figsize=(FULL_COL, 7.0), sharex=True)
for ax, f in zip(axes, FF5_FACTORS):
    s = P[f]
    ax.plot(s.index, s.values, color=FACTOR_PALETTE[f], linewidth=0.9)
    ax.axhline(0.5, color=C['muted'], linewidth=0.4, linestyle='--')
    ax.set_ylim(0.30, 0.70)
    ax.set_ylabel(f, color=FACTOR_PALETTE[f], fontsize=10, fontweight='bold')
    ax.set_yticks([0.35, 0.5, 0.65])
axes[0].set_title('Method B: Ridge-implied P(favorable), 1990 to 2026',
                  loc='left', color=C['ink'])
yearly_xticks(axes[-1], every=5)
save_fig(fig, '08_method_b_probs_timeseries', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/08_method_b_probs_timeseries.png


## 3. Build factor-timing strategies

In [4]:
rows = {}
timed_returns = {}
for f in FF5_FACTORS:
    p = P[f].dropna()
    # Use 'soft' mode for Ridge so the small probability deviations actually move weights
    w = prob_to_weight(p, mode='longflat')
    timed = apply_weights(w, panel[f], cost_bps_oneway=5.0)
    static = static_factor_returns(panel[f], timed.index)
    timed_returns[f] = timed['r_net']
    s_static = perf_stats(static)
    s_timed  = perf_stats(timed['r_net'])
    rows[f] = {
        'sharpe_static': s_static['sharpe_ann'],
        'sharpe_timed':  s_timed['sharpe_ann'],
        'mean_static':   s_static['mean_ann'],
        'mean_timed':    s_timed['mean_ann'],
        'vol_static':    s_static['vol_ann'],
        'vol_timed':     s_timed['vol_ann'],
        'mdd_static':    s_static['max_drawdown'],
        'mdd_timed':     s_timed['max_drawdown'],
    }
compare = pd.DataFrame(rows).T
compare['sharpe_uplift'] = compare['sharpe_timed'] - compare['sharpe_static']
save_table(compare, '07_method_b_per_factor_compare', out_dir=TABLE_DIR)
compare

  saved: ../tables/07_method_b_per_factor_compare.{csv,md}


,sharpe_static,sharpe_timed,mean_static,mean_timed,vol_static,vol_timed,mdd_static,mdd_timed,sharpe_uplift
MKT_RF,0.4748,0.4339,0.0744,0.0580,0.1567,0.1338,-0.5411,-0.4763,-0.0409
SMB,0.1191,0.1187,0.0121,0.0095,0.1017,0.0796,-0.3552,-0.3201,-0.0004
HML,0.2555,0.3856,0.0301,0.0356,0.1180,0.0922,-0.5779,-0.3199,0.1301
RMW,0.6411,0.4109,0.0584,0.0282,0.0912,0.0687,-0.2327,-0.1911,-0.2301
CMA,0.3501,0.4065,0.0269,0.0283,0.0769,0.0696,-0.2725,-0.2805,0.0564


## 4. Per-factor Sharpe bar chart

In [5]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.6))
x = np.arange(len(FF5_FACTORS))
w = 0.36
bars_s = ax.bar(x - w/2, compare['sharpe_static'].values, w, color=C['blue'],
                label='Static factor', edgecolor='white', linewidth=0.5)
bars_t = ax.bar(x + w/2, compare['sharpe_timed'].values, w, color=C['purple'],
                label='Method B timed', edgecolor='white', linewidth=0.5)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(FF5_FACTORS, fontsize=10)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Method B: timed vs static Sharpe, by factor (1990 to 2026)',
             loc='left', color=C['ink'])
ax.set_ylim(min(compare[['sharpe_static','sharpe_timed']].min().min(), 0) - 0.10,
            compare[['sharpe_static','sharpe_timed']].max().max() + 0.20)
bar_value_labels(ax, bars_s, fmt='{:+.2f}', offset=0.03, fontsize=8, color=C['blue'])
bar_value_labels(ax, bars_t, fmt='{:+.2f}', offset=0.03, fontsize=8, color=C['purple'])
legend_below(ax, ncol=2)
save_fig(fig, '09_method_b_sharpe_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/09_method_b_sharpe_bars.png


## 5. Composite portfolio (equal-weight across 5 timed sleeves)

In [6]:
tr_df = pd.DataFrame(timed_returns).dropna(how='all')
method_b_composite = tr_df.mean(axis=1)
static_df = pd.DataFrame({f: panel[f].shift(-1).reindex(tr_df.index) for f in FF5_FACTORS}).dropna(how='all')
static_composite = static_df.mean(axis=1)
stats = pd.DataFrame({
    'static_eq_weight': perf_stats(static_composite),
    'method_b_timed':   perf_stats(method_b_composite),
}).T
save_table(stats, '08_method_b_composite_perf', out_dir=TABLE_DIR)
stats

  saved: ../tables/08_method_b_composite_perf.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
static_eq_weight,312.0000,0.0404,0.0545,0.7417,-0.2192,2.0881,-0.1508
method_b_timed,312.0000,0.0319,0.0408,0.7813,0.2127,1.3425,-0.0910


In [7]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.4))
for r, color, label, lw in [
    (static_composite, C['blue'],   'Equal-weight static FF5', 1.0),
    (method_b_composite, C['purple'], 'Method B composite (timed)', 1.4),
]:
    cum = (1 + r).cumprod()
    ax.plot(cum.index, cum.values, color=color, linewidth=lw, label=label)
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Method B composite vs static FF5, 1990 to 2026',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=5)
legend_below(ax, ncol=2)
save_fig(fig, '10_method_b_composite_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/10_method_b_composite_cumret.png


In [8]:
P.to_csv(f'{TABLE_DIR}/09_method_b_probs.csv')
tr_df.to_csv(f'{TABLE_DIR}/10_method_b_timed_returns.csv')
print(f'Saved Method B outputs.')

Saved Method B outputs.
